# VitalSense - Pipeline Accuracy & Signal Quality Evaluation

**Contactless Physiological Monitoring System**

This notebook demonstrates the accuracy and reliability of VitalSense's core algorithms through live webcam evaluation:

| Component | Algorithm | Reference |
|---|---|---|
| Face Detection | MediaPipe Face Landmarker (478 landmarks) | Google MediaPipe |
| rPPG Signal | POS + CHROM + GREEN fusion | Wang et al. 2017, De Haan & Jeanne 2013 |
| Heart Rate | Welch FFT + Adaptive Peak Detection | rPPG-Toolbox |
| HRV | RMSSD, SDNN, pNN50, Lomb-Scargle LF/HF | Task Force ESC/NASPE 1996 |
| Eye Analysis | EAR blink detection, iris gaze tracking | Soukupova & Cech 2016 |
| Respiration | Landmark motion + BVP-derived RSA | Dual-channel fusion |
| Stress/Fatigue | Multi-factor physiological scoring | Clinical literature-based |

In [ ]:
import os, sys, time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from collections import Counter, deque
from IPython.display import display, HTML, clear_output

# Pipeline imports
from face_detection import FaceDetector, compute_geometric_embedding
from roi_extraction import extract_roi_signals, get_combined_rgb
from ippg_signal import (IPPGSignalExtractor, _pos_from_rgb, _chrom_from_rgb,
                         _green_from_rgb, _spectral_snr)
from signal_processing import (butterworth_bandpass, estimate_hr_fft,
                               detect_peaks, compute_heart_rate)
from hrv_analysis import compute_hrv, compute_ibi, compute_frequency_domain
from eye_analysis import EyeAnalyzer
from respiration_detection import RespirationDetector
from metrics_engine import (compute_stress_score, compute_cognitive_load,
                            compute_fatigue_risk, compute_mood)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print('All pipeline components loaded successfully!')

In [ ]:
# ============================================================
#  LIVE WEBCAM CAPTURE + FULL PIPELINE PROCESSING
# ============================================================

CAPTURE_DURATION = 30   # seconds
TARGET_FPS = 15

# Initialise pipeline
detector = FaceDetector(max_faces=1)
ippg = IPPGSignalExtractor(buffer_seconds=CAPTURE_DURATION + 5, fps=TARGET_FPS)
eye_analyzer = EyeAnalyzer(fps=TARGET_FPS, buffer_seconds=CAPTURE_DURATION + 5)
resp_detector = RespirationDetector(buffer_seconds=CAPTURE_DURATION + 5, fps=TARGET_FPS)

# Data collection
data = {
    'detection_results': [], 'detection_times_ms': [], 'landmark_counts': [],
    'rgb_r': [], 'rgb_g': [], 'rgb_b': [],
    'snr_pos': [], 'snr_chrom': [], 'snr_fused': [], 'methods': [],
    'hr_values': [], 'hr_history': [],
    'ear_values': [], 'blink_rates': [], 'gaze_stability': [], 'perclos': [],
    'rr_values': [],
    'stress': [], 'fatigue': [], 'cognitive': [], 'mood': [],
    'timestamps': [],
    'pos_signals': [], 'chrom_signals': [], 'green_signals': [],
}
ema_state = {}
hrv_result = {}

# Open webcam
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    raise RuntimeError('Cannot open webcam')

print(f'Capturing {CAPTURE_DURATION}s of webcam data... Sit still and face the camera.')
print('Press Q in the video window to stop early.\n')

start = time.time()
frame_count = 0
frame_interval = 1.0 / TARGET_FPS

while True:
    elapsed = time.time() - start
    if elapsed >= CAPTURE_DURATION:
        break

    ret, frame = cap.read()
    if not ret:
        continue

    frame_count += 1
    data['timestamps'].append(elapsed)

    # Progress
    if frame_count % 15 == 0:
        clear_output(wait=True)
        pct = elapsed / CAPTURE_DURATION * 100
        bar = '#' * int(pct // 2) + '-' * (50 - int(pct // 2))
        print(f'[{bar}] {pct:.0f}%  |  {frame_count} frames  |  {elapsed:.1f}s')

    # Face detection
    t0 = time.perf_counter()
    landmarks = detector.detect(frame)
    dt_ms = (time.perf_counter() - t0) * 1000
    data['detection_times_ms'].append(dt_ms)
    detected = landmarks is not None
    data['detection_results'].append(detected)

    if not detected:
        cv2.imshow('Capture', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    data['landmark_counts'].append(len(landmarks))

    # ROI + rPPG
    roi = extract_roi_signals(frame, landmarks)
    combined_rgb = get_combined_rgb(roi)

    if combined_rgb is not None:
        data['rgb_r'].append(combined_rgb[0])
        data['rgb_g'].append(combined_rgb[1])
        data['rgb_b'].append(combined_rgb[2])
        ippg.add_sample(combined_rgb)

        bvp = ippg.compute_chrom()
        data['snr_pos'].append(ippg.last_snr_pos)
        data['snr_chrom'].append(ippg.last_snr_chrom)
        data['snr_fused'].append(ippg.signal_quality_score)
        data['methods'].append(ippg.last_method)

        if len(bvp) > 30:
            # Individual algorithm signals for comparison
            rgb_arr = np.column_stack([np.array(ippg.r_buffer),
                                       np.array(ippg.g_buffer),
                                       np.array(ippg.b_buffer)])

            fft_hr = estimate_hr_fft(bvp, TARGET_FPS)
            peaks = detect_peaks(bvp, TARGET_FPS)
            hr = compute_heart_rate(peaks, TARGET_FPS, data['hr_history'], fft_hr)
            if hr > 0:
                data['hr_values'].append(hr)
                data['hr_history'].append(hr)
            hrv_result = compute_hrv(peaks, TARGET_FPS)
            resp_detector.add_bvp_signal(bvp)

    # Eye analysis
    eye_analyzer.add_landmarks(landmarks)
    em = eye_analyzer.get_metrics()
    data['ear_values'].append(em.get('ear_avg', 0))
    data['blink_rates'].append(em.get('blink_rate', 0))
    data['gaze_stability'].append(em.get('gaze_stability', 0))
    data['perclos'].append(em.get('perclos', 0))

    # Respiration
    resp_detector.add_landmarks(landmarks)
    rr = resp_detector.compute_respiration_rate()
    if rr > 0:
        data['rr_values'].append(rr)

    # Derived metrics
    if len(data['hr_values']) > 5:
        last_hr = data['hr_values'][-1]
        last_rr = data['rr_values'][-1] if data['rr_values'] else 0

        s = compute_stress_score(last_hr, hrv_result.get('rmssd', 0), last_rr,
                                 lf_hf_ratio=hrv_result.get('lf_hf_ratio', 0),
                                 pnn50=hrv_result.get('pnn50', -1),
                                 blink_rate=em.get('blink_rate', -1),
                                 saccade_rate=em.get('saccade_rate', -1),
                                 gaze_stability=em.get('gaze_stability', -1),
                                 ema_state=ema_state)
        data['stress'].append(s)

        f = compute_fatigue_risk(last_hr, hrv_result.get('rmssd', 0), hrv_result.get('sdnn', 0),
                                last_rr, hr_history=data['hr_history'],
                                pnn50=hrv_result.get('pnn50', -1),
                                lf_hf_ratio=hrv_result.get('lf_hf_ratio', 0),
                                blink_rate=em.get('blink_rate', -1),
                                perclos=em.get('perclos', -1),
                                ear_avg=em.get('ear_avg', -1),
                                gaze_stability=em.get('gaze_stability', -1),
                                ema_state=ema_state)
        data['fatigue'].append(f)

        c = compute_cognitive_load(last_hr, hrv_result.get('rmssd', 0), last_rr,
                                  hr_history=data['hr_history'],
                                  lf_hf_ratio=hrv_result.get('lf_hf_ratio', 0),
                                  blink_rate=em.get('blink_rate', -1),
                                  gaze_stability=em.get('gaze_stability', -1),
                                  saccade_rate=em.get('saccade_rate', -1),
                                  pupil_size=em.get('pupil_size', -1))
        data['cognitive'].append(c)

        m = compute_mood(s, hrv_result.get('rmssd', 0), last_hr,
                         lf_hf_ratio=hrv_result.get('lf_hf_ratio', 0),
                         pnn50=hrv_result.get('pnn50', -1),
                         blink_rate=em.get('blink_rate', -1),
                         gaze_stability=em.get('gaze_stability', -1))
        data['mood'].append(m)

    cv2.imshow('Capture', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    sleep_t = frame_interval - (time.perf_counter() - t0)
    if sleep_t > 0:
        time.sleep(sleep_t)

cap.release()
cv2.destroyAllWindows()
total_time = time.time() - start
actual_fps = frame_count / total_time

clear_output(wait=True)
print(f'Capture complete: {frame_count} frames in {total_time:.1f}s ({actual_fps:.1f} FPS)')
print(f'Detected faces in {sum(data["detection_results"])}/{len(data["detection_results"])} frames')
print(f'Collected {len(data["hr_values"])} HR readings, {len(data["rr_values"])} RR readings')

# Store final BVP and algorithms
bvp_final = ippg.get_signal()
peaks_final = detect_peaks(bvp_final, TARGET_FPS) if len(bvp_final) > 30 else np.array([])
hrv_final = compute_hrv(peaks_final, TARGET_FPS) if len(peaks_final) > 2 else {}

# Get individual algorithm signals for comparison
rgb_full = np.column_stack([np.array(ippg.r_buffer),
                            np.array(ippg.g_buffer),
                            np.array(ippg.b_buffer)])
pos_sig = _pos_from_rgb(rgb_full, TARGET_FPS) if len(rgb_full) > 10 else np.array([])
chrom_sig = _chrom_from_rgb(rgb_full, TARGET_FPS) if len(rgb_full) > 10 else np.array([])
green_sig = _green_from_rgb(rgb_full) if len(rgb_full) > 10 else np.array([])

detector.close()

## 1. Face Detection Accuracy
MediaPipe Face Landmarker detects 478 facial landmarks per face with sub-millisecond processing.

In [ ]:
det_rate = sum(data['detection_results']) / len(data['detection_results']) * 100
avg_time = np.mean(data['detection_times_ms'])
avg_lm = np.mean(data['landmark_counts']) if data['landmark_counts'] else 0

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Detection rate pie
detected = sum(data['detection_results'])
missed = len(data['detection_results']) - detected
axes[0].pie([detected, missed], labels=['Detected', 'Missed'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, explode=(0.05, 0))
axes[0].set_title(f'Face Detection Rate\n({detected}/{len(data["detection_results"])} frames)', fontweight='bold')

# Processing time histogram
axes[1].hist(data['detection_times_ms'], bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[1].axvline(avg_time, color='red', linestyle='--', label=f'Mean: {avg_time:.1f} ms')
axes[1].set_xlabel('Processing Time (ms)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Detection Speed Distribution', fontweight='bold')
axes[1].legend()

# Landmark count
axes[2].bar(['Detected', 'Expected'], [avg_lm, 478], color=['#2ecc71', '#bdc3c7'],
            edgecolor='white', linewidth=1.5)
axes[2].set_title('Landmarks Per Face', fontweight='bold')
axes[2].set_ylabel('Count')
for i, v in enumerate([avg_lm, 478]):
    axes[2].text(i, v + 5, f'{v:.0f}', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('evaluation_results/01_face_detection.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nDetection Rate  : {det_rate:.1f}%')
print(f'Avg Latency     : {avg_time:.1f} ms/frame')
print(f'Avg Landmarks   : {avg_lm:.0f} / 478')
print(f'Verdict         : {"PASS" if det_rate > 90 else "NEEDS IMPROVEMENT"}')

## 2. rPPG Signal Quality Analysis
Three research-grade algorithms are compared: **POS** (Wang 2017), **CHROM** (De Haan 2013), and **GREEN** (Verkruysse 2008). The system uses SNR-weighted fusion of POS+CHROM.

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(3, 2, figure=fig, hspace=0.4, wspace=0.3)

# Raw RGB channels
ax1 = fig.add_subplot(gs[0, 0])
if data['rgb_r']:
    t = np.arange(len(data['rgb_r'])) / TARGET_FPS
    ax1.plot(t, data['rgb_r'], 'r-', alpha=0.7, linewidth=0.8, label='Red')
    ax1.plot(t, data['rgb_g'], 'g-', alpha=0.7, linewidth=0.8, label='Green')
    ax1.plot(t, data['rgb_b'], 'b-', alpha=0.7, linewidth=0.8, label='Blue')
    ax1.legend(fontsize=8)
ax1.set_title('Raw RGB Channels from Skin ROI', fontweight='bold')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Intensity')

# Individual algorithm outputs
ax2 = fig.add_subplot(gs[0, 1])
if len(pos_sig) > 0:
    t = np.arange(len(pos_sig)) / TARGET_FPS
    # Normalize for comparison
    ps = (pos_sig - np.mean(pos_sig)) / (np.std(pos_sig) + 1e-10)
    cs = (chrom_sig[:len(pos_sig)] - np.mean(chrom_sig[:len(pos_sig)])) / (np.std(chrom_sig[:len(pos_sig)]) + 1e-10)
    gs_sig = (green_sig[:len(pos_sig)] - np.mean(green_sig[:len(pos_sig)])) / (np.std(green_sig[:len(pos_sig)]) + 1e-10)
    ax2.plot(t, ps + 6, color='#e74c3c', linewidth=0.8, label='POS')
    ax2.plot(t, cs + 3, color='#2ecc71', linewidth=0.8, label='CHROM')
    ax2.plot(t, gs_sig, color='#3498db', linewidth=0.8, label='GREEN')
    ax2.legend(fontsize=8)
ax2.set_title('Algorithm Output Comparison (normalized)', fontweight='bold')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Amplitude (offset for clarity)')

# SNR comparison bar chart
ax3 = fig.add_subplot(gs[1, 0])
snr_vals = [
    _spectral_snr(pos_sig, TARGET_FPS) if len(pos_sig) > 15 else 0,
    _spectral_snr(chrom_sig, TARGET_FPS) if len(chrom_sig) > 15 else 0,
    _spectral_snr(green_sig, TARGET_FPS) if len(green_sig) > 15 else 0,
    _spectral_snr(bvp_final, TARGET_FPS) if len(bvp_final) > 15 else 0,
]
labels = ['POS', 'CHROM', 'GREEN', 'Fused\n(POS+CHROM)']
colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']
bars = ax3.bar(labels, snr_vals, color=colors, edgecolor='white', linewidth=2)
ax3.axhline(y=2.0, color='orange', linestyle='--', linewidth=1.5, label='Min Quality (2.0)')
for b, v in zip(bars, snr_vals):
    ax3.text(b.get_x() + b.get_width()/2, b.get_height() + 0.1,
             f'{v:.2f}', ha='center', fontweight='bold', fontsize=11)
ax3.set_title('Signal-to-Noise Ratio Comparison', fontweight='bold')
ax3.set_ylabel('SNR')
ax3.legend()

# SNR over time
ax4 = fig.add_subplot(gs[1, 1])
if data['snr_pos']:
    ax4.plot(data['snr_pos'], color='#e74c3c', alpha=0.6, linewidth=0.8, label='POS')
    ax4.plot(data['snr_chrom'], color='#2ecc71', alpha=0.6, linewidth=0.8, label='CHROM')
    ax4.plot(data['snr_fused'], color='#9b59b6', linewidth=1.5, label='Fused (max)')
    ax4.axhline(y=2.0, color='orange', linestyle='--', label='Min Threshold')
    ax4.legend(fontsize=8)
ax4.set_title('Signal Quality Over Time', fontweight='bold')
ax4.set_xlabel('Sample')
ax4.set_ylabel('SNR')

# Fused BVP signal
ax5 = fig.add_subplot(gs[2, 0])
if len(bvp_final) > 0:
    t = np.arange(len(bvp_final)) / TARGET_FPS
    ax5.plot(t, bvp_final, color='#e74c3c', linewidth=1)
    if len(peaks_final) > 0:
        ax5.plot(peaks_final / TARGET_FPS, bvp_final[peaks_final], 'gv',
                 markersize=6, label=f'{len(peaks_final)} peaks detected')
        ax5.legend(fontsize=8)
ax5.set_title('Fused BVP Signal with Detected Heartbeats', fontweight='bold')
ax5.set_xlabel('Time (s)')
ax5.set_ylabel('Amplitude')

# Method selection
ax6 = fig.add_subplot(gs[2, 1])
method_c = Counter(data['methods'])
if method_c:
    ax6.pie(method_c.values(), labels=method_c.keys(), autopct='%1.1f%%',
            colors=['#9b59b6', '#2ecc71', '#f39c12'], startangle=90)
ax6.set_title('Algorithm Selection Distribution', fontweight='bold')

plt.savefig('evaluation_results/02_rppg_quality.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nFinal SNR - POS: {snr_vals[0]:.2f} | CHROM: {snr_vals[1]:.2f} | GREEN: {snr_vals[2]:.2f} | Fused: {snr_vals[3]:.2f}')
print(f'Quality: {"Excellent" if snr_vals[3] > 5 else "Good" if snr_vals[3] > 3 else "Acceptable" if snr_vals[3] > 2 else "Low"}')

## 3. Heart Rate Analysis
HR is estimated using a hybrid approach: **Welch FFT** (primary) + **adaptive peak detection** (secondary), with EMA smoothing.

In [ ]:
hr_arr = np.array(data['hr_values'])

if len(hr_arr) > 0:
    hr_mean = np.mean(hr_arr)
    hr_std = np.std(hr_arr)
    hr_cv = hr_std / hr_mean * 100
    hr_plausible = np.sum((hr_arr >= 50) & (hr_arr <= 120)) / len(hr_arr) * 100

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # HR over time
    axes[0].plot(hr_arr, color='#e74c3c', linewidth=1.5)
    axes[0].axhline(hr_mean, color='#3498db', linestyle='--', label=f'Mean: {hr_mean:.1f}')
    axes[0].fill_between(range(len(hr_arr)), hr_mean - hr_std, hr_mean + hr_std,
                         alpha=0.15, color='#3498db')
    axes[0].axhspan(60, 100, alpha=0.08, color='green', label='Normal (60-100)')
    axes[0].set_title('Heart Rate Over Time', fontweight='bold')
    axes[0].set_xlabel('Sample')
    axes[0].set_ylabel('BPM')
    axes[0].legend(fontsize=8)

    # HR distribution
    axes[1].hist(hr_arr, bins=20, color='#e74c3c', edgecolor='white', alpha=0.8)
    axes[1].axvline(hr_mean, color='blue', linestyle='--', label=f'Mean: {hr_mean:.1f}')
    axes[1].axvspan(60, 100, alpha=0.1, color='green', label='Normal Range')
    axes[1].set_title('HR Distribution', fontweight='bold')
    axes[1].set_xlabel('BPM')
    axes[1].set_ylabel('Count')
    axes[1].legend(fontsize=8)

    # Stability metrics
    metrics = ['Mean HR\n(BPM)', 'Std Dev\n(BPM)', 'CV\n(%)', 'Plausible\n(%)']
    values = [hr_mean, hr_std, hr_cv, hr_plausible]
    colors = ['#3498db', '#f39c12',
              '#2ecc71' if hr_cv < 15 else '#e74c3c',
              '#2ecc71' if hr_plausible > 90 else '#e74c3c']
    bars = axes[2].bar(metrics, values, color=colors, edgecolor='white', linewidth=1.5)
    for b, v in zip(bars, values):
        axes[2].text(b.get_x() + b.get_width()/2, b.get_height() + 0.5,
                     f'{v:.1f}', ha='center', fontweight='bold', fontsize=10)
    axes[2].set_title('HR Stability Metrics', fontweight='bold')
    axes[2].set_ylabel('Value')

    plt.tight_layout()
    plt.savefig('evaluation_results/03_heart_rate.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\nMean HR: {hr_mean:.1f} BPM (SD: {hr_std:.1f}, CV: {hr_cv:.1f}%)')
    print(f'Plausibility: {hr_plausible:.1f}% readings in physiological range (50-120 BPM)')
    print(f'Stability: {"Excellent" if hr_cv < 5 else "Good" if hr_cv < 10 else "Acceptable" if hr_cv < 15 else "Variable"}')
else:
    print('No heart rate data collected. Ensure face is visible for full capture duration.')

## 4. HRV Analysis (Time + Frequency Domain)
HRV metrics computed from inter-beat intervals using Lomb-Scargle periodogram for frequency analysis.

In [ ]:
if hrv_final:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Time-domain metrics with normal ranges
    td_labels = ['RMSSD (ms)', 'SDNN (ms)', 'pNN50 (%)']
    td_values = [hrv_final.get('rmssd', 0), hrv_final.get('sdnn', 0), hrv_final.get('pnn50', 0)]
    td_normals = [(20, 60), (30, 100), (5, 25)]
    td_colors = []
    for v, (lo, hi) in zip(td_values, td_normals):
        td_colors.append('#2ecc71' if lo <= v <= hi else '#f39c12' if v > 0 else '#bdc3c7')

    bars = axes[0].bar(td_labels, td_values, color=td_colors, edgecolor='white', linewidth=1.5)
    for b, v, (lo, hi) in zip(bars, td_values, td_normals):
        axes[0].text(b.get_x() + b.get_width()/2, b.get_height() + 0.5,
                     f'{v:.1f}', ha='center', fontweight='bold', fontsize=11)
    axes[0].set_title('Time-Domain HRV (Green = Normal)', fontweight='bold')
    axes[0].set_ylabel('Value')

    # LF/HF visual
    lf = hrv_final.get('lf_power', 0)
    hf = hrv_final.get('hf_power', 0)
    ratio = hrv_final.get('lf_hf_ratio', 0)
    if lf > 0 or hf > 0:
        axes[1].bar(['LF Power\n(Sympathetic)', 'HF Power\n(Parasympathetic)'],
                    [lf, hf], color=['#e74c3c', '#3498db'], edgecolor='white', linewidth=1.5)
        axes[1].set_title(f'Frequency-Domain HRV\nLF/HF Ratio: {ratio:.2f} (Normal: 0.5-2.0)',
                         fontweight='bold')
        axes[1].set_ylabel('Power')
    else:
        axes[1].text(0.5, 0.5, 'Insufficient IBI data', ha='center', transform=axes[1].transAxes)

    # IBI series if available
    ibis = compute_ibi(peaks_final, TARGET_FPS)
    if len(ibis) > 2:
        axes[2].plot(ibis, 'o-', color='#9b59b6', markersize=4, linewidth=1)
        axes[2].axhline(np.mean(ibis), color='gray', linestyle='--',
                        label=f'Mean IBI: {np.mean(ibis):.0f} ms')
        axes[2].set_title(f'Inter-Beat Intervals ({len(ibis)} detected)', fontweight='bold')
        axes[2].set_xlabel('Beat Number')
        axes[2].set_ylabel('IBI (ms)')
        axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig('evaluation_results/04_hrv_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\nRMSSD: {hrv_final.get("rmssd", 0):.1f} ms (Normal: 20-60 ms)')
    print(f'SDNN : {hrv_final.get("sdnn", 0):.1f} ms (Normal: 30-100 ms)')
    print(f'pNN50: {hrv_final.get("pnn50", 0):.1f}% (Normal: 5-25%)')
    print(f'LF/HF: {hrv_final.get("lf_hf_ratio", 0):.2f} (Normal: 0.5-2.0)')
else:
    print('Insufficient heartbeat data for HRV analysis.')

## 5. Eye Analysis (EAR, Blink, Gaze, PERCLOS)
Eye Analysis uses MediaPipe iris landmarks (468-477) and the Eye Aspect Ratio formula from Soukupova & Cech (2016).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# EAR over time
valid_ear = [e for e in data['ear_values'] if e > 0]
if valid_ear:
    axes[0, 0].plot(valid_ear, color='#9b59b6', linewidth=0.8, alpha=0.8)
    axes[0, 0].axhline(0.24, color='red', linestyle='--', linewidth=1, label='Blink Threshold (0.24)')
    axes[0, 0].axhspan(0.25, 0.35, alpha=0.1, color='green', label='Normal Open Range')
    axes[0, 0].legend(fontsize=7)
axes[0, 0].set_title('Eye Aspect Ratio (EAR) Over Time', fontweight='bold')
axes[0, 0].set_ylabel('EAR')

# Gaze stability over time
valid_gs = [g for g in data['gaze_stability'] if g > 0]
if valid_gs:
    axes[0, 1].plot(valid_gs, color='#1abc9c', linewidth=1)
    axes[0, 1].axhline(np.mean(valid_gs), color='gray', linestyle='--',
                        label=f'Mean: {np.mean(valid_gs):.2f}')
axes[0, 1].set_title('Gaze Stability Over Time', fontweight='bold')
axes[0, 1].set_ylabel('Stability (0-1)')
axes[0, 1].set_ylim(0, 1.05)
axes[0, 1].legend(fontsize=8)

# Blink rate
valid_br = [b for b in data['blink_rates'] if b > 0]
if valid_br:
    axes[1, 0].plot(valid_br, color='#e67e22', linewidth=1)
    axes[1, 0].axhspan(15, 20, alpha=0.1, color='green', label='Normal (15-20 blinks/min)')
    axes[1, 0].legend(fontsize=8)
axes[1, 0].set_title('Blink Rate Over Time', fontweight='bold')
axes[1, 0].set_ylabel('Blinks/min')
axes[1, 0].set_xlabel('Sample')

# Summary metrics
axes[1, 1].axis('off')
ear_mean_val = np.mean(valid_ear) if valid_ear else 0
gs_mean_val = np.mean(valid_gs) if valid_gs else 0
br_last = data['blink_rates'][-1] if data['blink_rates'] else 0
perclos_last = data['perclos'][-1] if data['perclos'] else 0

summary = [
    ['Metric', 'Value', 'Normal Range', 'Status'],
    ['Avg EAR', f'{ear_mean_val:.3f}', '0.25 - 0.35', 'PASS' if 0.15 < ear_mean_val < 0.45 else 'WARN'],
    ['Blink Rate', f'{br_last:.1f}/min', '15 - 20/min', 'OK'],
    ['Gaze Stability', f'{gs_mean_val:.2f}', '> 0.6', 'PASS' if gs_mean_val > 0.5 else 'WARN'],
    ['PERCLOS', f'{perclos_last:.1f}%', '< 15%', 'PASS' if perclos_last < 20 else 'WARN'],
]
table = axes[1, 1].table(cellText=summary[1:], colLabels=summary[0],
                         cellLoc='center', loc='center', colWidths=[0.25, 0.2, 0.25, 0.15])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)
for j in range(4):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')
axes[1, 1].set_title('Eye Analysis Summary', fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('evaluation_results/05_eye_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Respiration Rate Analysis
Dual-channel approach: facial landmark micro-movements (nose, chin, jaw) + BVP-derived Respiratory Sinus Arrhythmia (RSA).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Respiratory waveform
resp_wave = resp_detector.get_waveform(200)
if resp_wave:
    t = np.arange(len(resp_wave)) / TARGET_FPS
    axes[0].plot(t, resp_wave, color='#1abc9c', linewidth=1.5)
    axes[0].set_xlabel('Time (s)')
axes[0].set_title('Respiratory Waveform', fontweight='bold')
axes[0].set_ylabel('Amplitude')

# RR over time + stats
if data['rr_values']:
    rr_arr = np.array(data['rr_values'])
    rr_mean = np.mean(rr_arr)
    rr_std = np.std(rr_arr)
    rr_plausible = np.sum((rr_arr >= 8) & (rr_arr <= 25)) / len(rr_arr) * 100

    axes[1].plot(rr_arr, color='#1abc9c', linewidth=1.5)
    axes[1].axhline(rr_mean, color='gray', linestyle='--', label=f'Mean: {rr_mean:.1f}')
    axes[1].fill_between(range(len(rr_arr)), rr_mean - rr_std, rr_mean + rr_std,
                         alpha=0.15, color='#1abc9c')
    axes[1].axhspan(12, 20, alpha=0.08, color='green', label='Normal (12-20)')
    axes[1].legend(fontsize=8)
    axes[1].set_xlabel('Sample')
    axes[1].set_ylabel('Breaths/min')
    axes[1].set_title('Respiration Rate Over Time', fontweight='bold')

    print(f'Mean RR: {rr_mean:.1f} breaths/min (SD: {rr_std:.1f})')
    print(f'Plausibility: {rr_plausible:.1f}% in physiological range (8-25)')
else:
    axes[1].text(0.5, 0.5, 'RR data still accumulating\n(needs ~10s of stable signal)',
                ha='center', va='center', transform=axes[1].transAxes, fontsize=12)

plt.tight_layout()
plt.savefig('evaluation_results/06_respiration.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Derived Metrics (Stress, Fatigue, Cognitive Load, Mood)
Multi-factor heuristic scoring models using physiological signals + eye metrics, with clinical literature-based thresholds.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Stress over time
if data['stress']:
    axes[0, 0].plot(data['stress'], color='#e74c3c', linewidth=1.5)
    axes[0, 0].axhspan(0, 30, alpha=0.08, color='green', label='Low (0-30)')
    axes[0, 0].axhspan(30, 60, alpha=0.08, color='orange', label='Moderate (30-60)')
    axes[0, 0].axhspan(60, 100, alpha=0.08, color='red', label='High (60-100)')
    axes[0, 0].set_ylim(0, 100)
    axes[0, 0].legend(fontsize=7, loc='upper right')
axes[0, 0].set_title(f'Stress Score (Avg: {np.mean(data["stress"]):.1f}/100)' if data['stress'] else 'Stress Score',
                     fontweight='bold')
axes[0, 0].set_ylabel('Score (0-100)')

# Fatigue over time
if data['fatigue']:
    axes[0, 1].plot(data['fatigue'], color='#3498db', linewidth=1.5)
    axes[0, 1].axhspan(0, 30, alpha=0.08, color='green', label='Low Risk')
    axes[0, 1].axhspan(30, 60, alpha=0.08, color='orange', label='Moderate Risk')
    axes[0, 1].axhspan(60, 100, alpha=0.08, color='red', label='High Risk')
    axes[0, 1].set_ylim(0, 100)
    axes[0, 1].legend(fontsize=7, loc='upper right')
axes[0, 1].set_title(f'Fatigue Risk (Avg: {np.mean(data["fatigue"]):.1f}/100)' if data['fatigue'] else 'Fatigue Risk',
                     fontweight='bold')
axes[0, 1].set_ylabel('Risk (0-100)')

# Cognitive load distribution
if data['cognitive']:
    cog_c = Counter(data['cognitive'])
    labels_cog = list(cog_c.keys())
    vals_cog = list(cog_c.values())
    color_map = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
    cog_colors = [color_map.get(l, '#bdc3c7') for l in labels_cog]
    axes[1, 0].pie(vals_cog, labels=labels_cog, autopct='%1.1f%%',
                   colors=cog_colors, startangle=90)
axes[1, 0].set_title('Cognitive Load Distribution', fontweight='bold')

# Mood distribution
if data['mood']:
    mood_c = Counter(data['mood'])
    labels_m = list(mood_c.keys())
    vals_m = list(mood_c.values())
    mood_colors = {'Happy': '#2ecc71', 'Calm': '#3498db', 'Neutral': '#95a5a6',
                   'Anxious': '#f39c12', 'Stressed': '#e74c3c', 'Fatigued': '#8e44ad'}
    m_colors = [mood_colors.get(l, '#bdc3c7') for l in labels_m]
    axes[1, 1].pie(vals_m, labels=labels_m, autopct='%1.1f%%',
                   colors=m_colors, startangle=90)
axes[1, 1].set_title('Mood Classification Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig('evaluation_results/07_derived_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Algorithm Comparison & Processing Performance

In [ ]:
import timeit

# Benchmark individual algorithms
if len(rgb_full) > 30:
    n_runs = 5
    times = {}
    for name, func in [('POS', lambda: _pos_from_rgb(rgb_full, TARGET_FPS)),
                       ('CHROM', lambda: _chrom_from_rgb(rgb_full, TARGET_FPS)),
                       ('GREEN', lambda: _green_from_rgb(rgb_full))]:
        t = timeit.timeit(func, number=n_runs) / n_runs * 1000
        times[name] = t

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Processing time comparison
    algs = list(times.keys())
    ms = list(times.values())
    colors = ['#e74c3c', '#2ecc71', '#3498db']
    bars = axes[0].bar(algs, ms, color=colors, edgecolor='white', linewidth=2)
    for b, v in zip(bars, ms):
        axes[0].text(b.get_x() + b.get_width()/2, b.get_height() + 0.2,
                     f'{v:.1f} ms', ha='center', fontweight='bold', fontsize=11)
    axes[0].set_title(f'Algorithm Processing Time ({len(rgb_full)} samples)', fontweight='bold')
    axes[0].set_ylabel('Time (ms)')

    # SNR vs Speed scatter
    snrs = [snr_vals[0], snr_vals[1], snr_vals[2]]
    for i, (alg, s, t) in enumerate(zip(algs, snrs, ms)):
        axes[1].scatter(t, s, s=200, c=colors[i], edgecolors='black',
                        linewidth=1.5, zorder=3, label=alg)
        axes[1].annotate(alg, (t, s), textcoords='offset points',
                        xytext=(10, 5), fontweight='bold', fontsize=11)
    axes[1].axhline(2.0, color='orange', linestyle='--', alpha=0.7, label='Min SNR')
    axes[1].set_xlabel('Processing Time (ms)')
    axes[1].set_ylabel('Signal Quality (SNR)')
    axes[1].set_title('Quality vs Speed Trade-off', fontweight='bold')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig('evaluation_results/08_algorithm_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    for alg, t in times.items():
        print(f'{alg}: {t:.1f} ms')
else:
    print('Insufficient data for algorithm benchmark.')

## 9. Overall Accuracy Summary Dashboard

In [ ]:
# Compute all summary values
det_rate = sum(data['detection_results']) / len(data['detection_results']) * 100
hr_mean = np.mean(data['hr_values']) if data['hr_values'] else 0
hr_cv = (np.std(data['hr_values']) / hr_mean * 100) if hr_mean > 0 else 999
hr_plausible = (np.sum((np.array(data['hr_values']) >= 50) &
                       (np.array(data['hr_values']) <= 120)) /
                len(data['hr_values']) * 100) if data['hr_values'] else 0
snr_final = _spectral_snr(bvp_final, TARGET_FPS) if len(bvp_final) > 15 else 0
rr_mean = np.mean(data['rr_values']) if data['rr_values'] else 0
ear_m = np.mean([e for e in data['ear_values'] if e > 0]) if data['ear_values'] else 0

checks = [
    ('Face Detection Rate', f'{det_rate:.1f}%', '> 90%', det_rate > 90),
    ('Heart Rate (Mean)', f'{hr_mean:.1f} BPM', '50-120 BPM', 50 < hr_mean < 120 if hr_mean > 0 else False),
    ('HR Stability (CV)', f'{hr_cv:.1f}%', '< 15%', hr_cv < 15),
    ('HR Plausibility', f'{hr_plausible:.1f}%', '> 90%', hr_plausible > 90),
    ('rPPG Signal SNR', f'{snr_final:.2f}', '> 2.0', snr_final > 2.0),
    ('RMSSD', f'{hrv_final.get("rmssd", 0):.1f} ms', '10-80 ms', 10 < hrv_final.get('rmssd', 0) < 80 if hrv_final else False),
    ('Respiration Rate', f'{rr_mean:.1f}', '8-25 BPM', 8 < rr_mean < 25 if rr_mean > 0 else False),
    ('Avg EAR', f'{ear_m:.3f}', '0.15-0.45', 0.15 < ear_m < 0.45 if ear_m > 0 else False),
]

passed = sum(1 for _, _, _, ok in checks if ok)
total = len(checks)

fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')

header = ['Metric', 'Measured', 'Criteria', 'Result']
cell_text = [[name, val, crit, 'PASS' if ok else 'FAIL'] for name, val, crit, ok in checks]

table = ax.table(cellText=cell_text, colLabels=header,
                 cellLoc='center', loc='center',
                 colWidths=[0.3, 0.2, 0.2, 0.12])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2)

# Style header
for j in range(4):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Style results
for i, (_, _, _, ok) in enumerate(checks):
    cell = table[i + 1, 3]
    if ok:
        cell.set_text_props(color='#27ae60', fontweight='bold')
        cell.set_facecolor('#eafaf1')
    else:
        cell.set_text_props(color='#c0392b', fontweight='bold')
        cell.set_facecolor('#fdedec')

verdict = 'SYSTEM VALIDATED' if passed >= total - 1 else 'ACCEPTABLE' if passed >= total - 2 else 'NEEDS REVIEW'
color = '#27ae60' if passed >= total - 1 else '#f39c12' if passed >= total - 2 else '#c0392b'

ax.set_title(f'VitalSense Accuracy Evaluation Summary\n{passed}/{total} checks passed — {verdict}',
             fontweight='bold', fontsize=14, pad=30, color=color)

plt.savefig('evaluation_results/09_summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n{"=" * 50}')
print(f'  RESULT: {passed}/{total} checks passed')
print(f'  VERDICT: {verdict}')
print(f'{"=" * 50}')
print(f'\nAll charts saved to evaluation_results/ directory.')